In [3]:
import pandas as pd
import numpy as np
import requests
import joblib
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ==========================================
# 1. CẤU HÌNH HẰNG SỐ 
# ==========================================
LATITUDE = 21.0050  
LONGITUDE = 106.8333 
MODEL_PATH = 'ensemble_model_pm25.pkl'
RAW_DATA_PATH = 'data_dongmai_all_raw.csv'
OUTPUT_DAILY_CSV = 'bao_cao_tung_ngay_30days.csv'
OUTPUT_HOURLY_CSV = 'du_lieu_chi_tiet_30days.csv'

# Mốc 30 ngày đánh giá (Ví dụ: 10/03 đến 10/04)
FETCH_START_DATE = "2026-03-08" # Lùi 2 ngày để lấy đà Rolling 24h
EVAL_START_DATE = "2026-03-10"
END_DATE = "2026-04-10"

WEATHER_API_URL = "https://archive-api.open-meteo.com/v1/archive"
AIR_QUALITY_API_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
BASE_MET_FEATURES = ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'wind_u', 'wind_v']

# ==========================================
# 2. HÀM TRUY XUẤT VÀ XỬ LÝ
# ==========================================
def fetch_historical_data(start_date, end_date):
    """Lấy dữ liệu API vệ tinh"""
    try:
        w_params = {"latitude": LATITUDE, "longitude": LONGITUDE, "start_date": start_date, "end_date": end_date, "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m", "timezone": "Asia/Bangkok"}
        df_weather = pd.DataFrame(requests.get(WEATHER_API_URL, params=w_params).json()['hourly'])
        
        aq_params = {"latitude": LATITUDE, "longitude": LONGITUDE, "start_date": start_date, "end_date": end_date, "hourly": "pm2_5,nitrogen_dioxide,sulphur_dioxide,carbon_monoxide,ozone,aerosol_optical_depth", "timezone": "Asia/Bangkok"}
        df_aq = pd.DataFrame(requests.get(AIR_QUALITY_API_URL, params=aq_params).json()['hourly'])
        
        df_merged = pd.merge(df_weather, df_aq, on='time', how='inner').dropna()
        df_merged['time'] = pd.to_datetime(df_merged['time'])
        return df_merged
    except Exception as e:
        print(f"Lỗi API: {e}")
        return None

def generate_30_day_report():
    if not os.path.exists(MODEL_PATH) or not os.path.exists(RAW_DATA_PATH):
        print("Lỗi: Thiếu file mô hình hoặc dữ liệu gốc.")
        return

    # Tải mô hình
    model_data = joblib.load(MODEL_PATH)
    model = model_data['model']
    feature_names = model_data['feature_names']

    # Khởi tạo lại Scaler chuẩn
    df_base = pd.read_csv(RAW_DATA_PATH)
    df_base['time'] = pd.to_datetime(df_base['time'])
    for col in ['hour', 'day_of_week', 'month']:
        df_base[col] = getattr(df_base['time'].dt, col if col != 'day_of_week' else 'dayofweek')
    
    wind_dir_rad_base = np.radians(df_base['wind_direction_10m'])
    df_base['wind_u'] = -df_base['wind_speed_10m'] * np.sin(wind_dir_rad_base)
    df_base['wind_v'] = -df_base['wind_speed_10m'] * np.cos(wind_dir_rad_base)
    
    for feature in BASE_MET_FEATURES:
        df_base[f'{feature}_rolling_12h'] = df_base[feature].rolling(window=12, min_periods=1).mean()
        df_base[f'{feature}_rolling_24h'] = df_base[feature].rolling(window=24, min_periods=1).mean()
        
    scaler = MinMaxScaler().fit(df_base.dropna()[feature_names])

    # Xử lý dữ liệu 30 ngày
    print(f"Đang xử lý dữ liệu 30 ngày từ {EVAL_START_DATE} đến {END_DATE}...")
    df_raw = fetch_historical_data(FETCH_START_DATE, END_DATE)
    
    for col in ['hour', 'day_of_week', 'month']:
        df_raw[col] = getattr(df_raw['time'].dt, col if col != 'day_of_week' else 'dayofweek')
        
    wind_dir_rad = np.radians(df_raw['wind_direction_10m'])
    df_raw['wind_u'] = -df_raw['wind_speed_10m'] * np.sin(wind_dir_rad)
    df_raw['wind_v'] = -df_raw['wind_speed_10m'] * np.cos(wind_dir_rad)
    
    for feature in BASE_MET_FEATURES:
        df_raw[f'{feature}_rolling_12h'] = df_raw[feature].rolling(window=12, min_periods=1).mean()
        df_raw[f'{feature}_rolling_24h'] = df_raw[feature].rolling(window=24, min_periods=1).mean()
        
    df_eval = df_raw[df_raw['time'] >= pd.to_datetime(EVAL_START_DATE)].reset_index(drop=True)
    
    # Dự đoán
    df_features = df_eval[feature_names].copy()
    df_features[feature_names] = scaler.transform(df_features[feature_names])
    df_eval['AI_Predicted_PM25'] = model.predict(df_features)
    
    # Tạo cột Ngày (Date) để Groupby
    df_eval['Date'] = df_eval['time'].dt.date
    
    # TỔNG HỢP THEO TỪNG NGÀY
    daily_stats = []
    for date, group in df_eval.groupby('Date'):
        y_true_day = group['pm2_5'].values
        y_pred_day = group['AI_Predicted_PM25'].values
        mae_day = mean_absolute_error(y_true_day, y_pred_day)
        rmse_day = np.sqrt(mean_squared_error(y_true_day, y_pred_day))
        daily_stats.append({'Ngày': date, 'MAE': round(mae_day, 2), 'RMSE': round(rmse_day, 2)})
        
    df_daily = pd.DataFrame(daily_stats)
    df_daily.to_csv(OUTPUT_DAILY_CSV, index=False)
    
    # TỔNG HỢP TOÀN BỘ 30 NGÀY
    y_true_all = df_eval['pm2_5'].values
    y_pred_all = df_eval['AI_Predicted_PM25'].values
    overall_mae = mean_absolute_error(y_true_all, y_pred_all)
    overall_rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))
    
    mean_obs = np.mean(y_true_all)
    overall_ioa = 1 - (np.sum((y_pred_all - y_true_all)**2) / np.sum((np.abs(y_pred_all - mean_obs) + np.abs(y_true_all - mean_obs))**2))
    overall_nmb = np.sum(y_pred_all - y_true_all) / np.sum(y_true_all)

    print("\n" + "="*60)
    print(" BẢNG TỔNG HỢP ĐÁNH GIÁ 30 NGÀY LIÊN TỤC (TỔNG TOÀN CỤC)")
    print("="*60)
    print(f"Tổng số giờ mô phỏng  : {len(df_eval)} giờ")
    print(f"MAE Toàn cục          : {overall_mae:.2f} µg/m³")
    print(f"RMSE Toàn cục         : {overall_rmse:.2f} µg/m³")
    print(f"IOA (Độ đồng điệu)    : {overall_ioa:.4f} (Mục tiêu ~0.73 )")
    print(f"NMB (Độ lệch chuẩn)   : {overall_nmb:.4f}")
    print("="*60)
    print(f"Đã xuất báo cáo từng ngày ra file: {OUTPUT_DAILY_CSV}")

if __name__ == "__main__":
    generate_30_day_report()

Đang xử lý dữ liệu 30 ngày từ 2026-03-10 đến 2026-04-10...

 BẢNG TỔNG HỢP ĐÁNH GIÁ 30 NGÀY LIÊN TỤC (TỔNG TOÀN CỤC)
Tổng số giờ mô phỏng  : 768 giờ
MAE Toàn cục          : 7.17 µg/m³
RMSE Toàn cục         : 9.99 µg/m³
IOA (Độ đồng điệu)    : 0.8856 (Mục tiêu ~0.73 )
NMB (Độ lệch chuẩn)   : -0.0030
Đã xuất báo cáo từng ngày ra file: bao_cao_tung_ngay_30days.csv
